# Category segmentation for articles


In [0]:
# COMMAND ---------- 
# Cell 1 – Install & basic setup
# (Run once after cluster start; may take 1-2 minutes)
# %pip install --quiet --upgrade sentence-transformers scikit-learn umap-learn hdbscan==0.8.33
# dbutils.library.restartPython()   # <- restarts the Python context so the new libs are visible 

In [0]:
# dbutils.library.restartPython() 

In [0]:
from pyspark.sql import functions as F, types as T
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np, pandas as pd, re, unicodedata, string, pathlib

In [0]:
DATA_PATH         = "https://github.com/davidtaki/Online_Retail_Project/raw/main/stockcode_db.csv"
ENCODING          = "latin1"
SEP               = ";"        # the csv is ";"-separated
MIN_WORD_LEN      = 3          # drop very short tokens in cleaning
MODEL_NAME        = "all-MiniLM-L6-v2"
min_cluster_size  = 10         # clusters smaller than this -> H1_Other
conf_threshold    = 0.80       # probability < 0.80 -> H1_Other

# cluster granularity per level (tweakable)
n_clusters_lvl1, n_clusters_lvl2, n_clusters_lvl3 = 10, 30, 80   

Databricks Free Edition notebooks run on Spark Connect.
Connect still executes all DataFrame logic on the cluster, but a few interactive helpers are missing or partly-implemented.
According to the official limitation list, the classic dataframe.display() API is not available, and the same RPC pathway can trip up .show() in some cases 

In [0]:
raw_df = (
    spark.read
         .option("header", "true")
         .option("sep", SEP)
         .option("encoding", ENCODING)
         .csv(DATA_PATH)
         .select(
             F.trim(F.col("StockCode")).alias("StockCode"),
             F.trim(F.col("Description")).alias("Description")
         )
         .dropna(subset=["Description"])   
         .dropDuplicates(["StockCode"])    
)
#raw_df.show() not working in free edition ! 

In [0]:
def basic_clean(txt: str) -> str:
    txt = txt.lower()
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode()
    txt = re.sub(r"[{}]".format(re.escape(string.punctuation)), " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    txt = " ".join([w for w in txt.split() if len(w) >= MIN_WORD_LEN])
    return txt

clean_udf = F.udf(basic_clean, T.StringType())
clean_df = raw_df.withColumn("clean_desc", clean_udf("Description"))

In [0]:
model = SentenceTransformer(MODEL_NAME)
@F.pandas_udf("array<float>")
def embed_series(col: pd.Series) -> pd.Series:
    return pd.Series(model.encode(col.to_list(), show_progress_bar=True).tolist())

vect_df = clean_df.withColumn("emb", embed_series("clean_desc"))

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW articles_view AS
SELECT 
    a.StockCode,
    a.Description
FROM online_retail.sales_articles AS a

In [0]:
# Spark DataFrame beolvasása a korábbi TEMP VIEW-ból
articles_df = spark.table("articles_view")
articles_pd = articles_df.toPandas()
articles_pd.head()

In [0]:
vect_pd = articles_pd[["StockCode","Description"]]
# 2) Sentence-BERT embeddings on the driver
model = SentenceTransformer(MODEL_NAME)
vect_pd["emb"] = model.encode(
    vect_pd["Description"].tolist(), show_progress_bar=True
).tolist()

# 3) Stack into a single (rows, 384) matrix
emb_mat = np.vstack(vect_pd["emb"].values)

# 4) Helper → adds a label column for each level
def cluster_and_append(df_pd, emb, n_clusters, col_name):
    labels = AgglomerativeClustering(
        n_clusters=n_clusters, metric="cosine", linkage="average"
    ).fit_predict(emb)
    df_pd[col_name] = labels
    return df_pd

# 5) Run three times with different k
vect_pd = cluster_and_append(vect_pd, emb_mat, n_clusters_lvl1, "lvl1_id")
vect_pd = cluster_and_append(vect_pd, emb_mat, n_clusters_lvl2, "lvl2_id")
vect_pd = cluster_and_append(vect_pd, emb_mat, n_clusters_lvl3, "lvl3_id")

In [0]:
def id_to_label(df, id_col, prefix="H1_"):
    tfidf = TfidfVectorizer(max_features=3_000)
    tfidf.fit(df["Description"])
    labels = {}
    for cid, group in df.groupby(id_col):
        top_idx = tfidf.transform(group["Description"]).sum(axis=0).A1.argsort()[-5:][::-1]
        words   = [tfidf.get_feature_names_out()[i] for i in top_idx if len(tfidf.get_feature_names_out()[i])>=MIN_WORD_LEN]
        label   = prefix + (" ".join(words[:3]).upper().replace(" ", "_") or "MISC")
        labels[cid] = label
    return df[id_col].map(labels)

vect_pd["lvl1_label"] = id_to_label(vect_pd, "lvl1_id")
vect_pd["lvl2_label"] = id_to_label(vect_pd, "lvl2_id")
vect_pd["lvl3_label"] = id_to_label(vect_pd, "lvl3_id")

In [0]:
# Apply “H1_Other” fallback for tiny / low-confidence clusters
# (Confidence proxy =  cluster size / total size)
total = len(vect_pd)
lvl_other_mask = vect_pd.groupby("lvl3_id")["StockCode"].transform("count") < min_cluster_size
vect_pd.loc[lvl_other_mask, ["lvl1_label","lvl2_label","lvl3_label"]] = ["H1_OTHER"]*3

In [0]:
# Create final mapping Spark DF
map_pd = vect_pd[["StockCode","lvl1_label","lvl2_label","lvl3_label"]]
map_spark = (
    spark.createDataFrame(map_pd)
          .withColumnRenamed("lvl1_label","H1_lvl1")
          .withColumnRenamed("lvl2_label","H1_lvl2")
          .withColumnRenamed("lvl3_label","H1_lvl3")
)

# Quick sanity checks
display(map_spark.groupBy("H1_lvl1").count().orderBy(F.desc("count")))
display(map_spark.filter(F.col("H1_lvl1")=="H1_OTHER").count())

In [0]:
%sql
CREATE OR REPLACE TABLE online_retail.sales_hierarchie (
  StockCode      STRING,
  lvl1_label    STRING,
  lvl2_label    STRING,
  lvl3_label    STRING
)
COMMENT '3 level hierarchie_table';

In [0]:
(map_spark
  .write
  .format("delta")
  .mode("overwrite")          # or "append" if you’re adding rows
  .option("overwriteSchema", "true")   # keep life simple if you overwrite
  .saveAsTable("online_retail.sales_hierarchie")
)

In [0]:
%sql

SELECT  
      hier.*,
      art.Description
FROM online_retail.sales_hierarchie hier
LEFT JOIN online_retail.sales_articles art ON hier.StockCode = art.StockCode;